# Propensity Modeling & Segmentation
- Justin Wall, Data Scientist, Antler Industrial Supply

## Data Import

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_PATH = "../data/antler_industrial_accounts.csv"  # adjust to your path
df = pd.read_csv(DATA_PATH)

print(df.shape)
print("duplicate ids:", df.account_id.duplicated().sum())
print("nulls:", df.isna().sum().sum())
df.describe().T[["min", "50%", "mean", "max"]].round(2)

(12000, 20)
duplicate ids: 0
nulls: 0


,min,50%,mean,max
account_age_months,1.0,43.50,49.76,240.00
orders_12m,0.0,4.00,7.69,257.00
orders_last_90d,0.0,1.00,1.99,49.00
revenue_12m,0.0,5142.36,11497.18,435454.54
avg_order_value,0.0,1126.85,1458.13,28950.01
days_since_last_order,0.0,55.00,117.29,730.00
lifetime_orders,1.0,14.00,33.19,1691.00
consumables_revenue_share,0.0,0.53,0.53,1.00
email_open_rate_90d,0.0,0.40,0.41,0.99
portal_logins_90d,0.0,3.00,4.09,49.00


***
### **Notes**:
- 3 Regions
- 8 categories
- price, units sold, revenue
***

## Business Alignment

In [2]:
# 2a. Internal consistency: each should print True
checks = {
    "orders_90d <= orders_12m": (df.orders_last_90d <= df.orders_12m).all(),
    "orders_12m <= lifetime_orders": (df.orders_12m <= df.lifetime_orders).all(),
    "revenue == 0 iff orders == 0": ((df.revenue_12m == 0) == (df.orders_12m == 0)).all(),
    "AOV ~= revenue / orders": np.allclose(
        df.avg_order_value,
        (df.revenue_12m / df.orders_12m.replace(0, np.nan)).fillna(0),
        atol=0.02,
    ),
    "next_30d_revenue > 0 iff ordered": ((df.next_30d_revenue > 0) == (df.ordered_next_30d == 1)).all(),
    "no-order accts have recency > 365": (df.loc[df.orders_12m == 0, "days_since_last_order"] > 365).all(),
}
pd.Series(checks)

orders_90d <= orders_12m             True
orders_12m <= lifetime_orders        True
revenue == 0 iff orders == 0         True
AOV ~= revenue / orders              True
next_30d_revenue > 0 iff ordered     True
no-order accts have recency > 365    True
dtype: bool

In [3]:
# 2b. Scale vs. the email: 12K accounts, 38 reps, 25-30 touches/rep/week
reps = df.groupby("assigned_rep_id").size()
print("accounts:", len(df), "| reps:", len(reps), "| regions:", df.region.nunique())
print("accounts per rep (min/median/max):", reps.min(), reps.median(), reps.max())

cap_lo, cap_hi = 38 * 25, 38 * 30
print(f"weekly capacity: {cap_lo}-{cap_hi} touches = {cap_lo/len(df):.1%}-{cap_hi/len(df):.1%} of base")
print(f"base 30d purchase rate: {df.ordered_next_30d.mean():.1%}")

# Email says ~1/3 of accounts have < 3 orders
print("orders_12m < 3:", f"{(df.orders_12m < 3).mean():.1%}")
print("zero orders in 12m:", f"{(df.orders_12m == 0).mean():.1%}")

accounts: 12000 | reps: 38 | regions: 5
accounts per rep (min/median/max): 224 319.0 378
weekly capacity: 950-1140 touches = 7.9%-9.5% of base
base 30d purchase rate: 36.7%
orders_12m < 3: 35.4%
zero orders in 12m: 8.0%


In [4]:
# 2c. Do tiers, verticals, regions, and size behave the way sales would expect?
tier_order = ["Bronze", "Silver", "Gold", "Platinum"]
tiers = df.groupby("loyalty_tier").agg(
    n=("account_id", "size"),
    purchase_rate=("ordered_next_30d", "mean"),
    median_rev=("revenue_12m", "median"),
    median_orders=("orders_12m", "median"),
).loc[tier_order]
tiers["share"] = tiers.n / len(df)
display(tiers.round(3))

for col in ["industry_vertical", "region", "employee_band"]:
    display(
        df.groupby(col)
        .agg(n=("account_id", "size"), purchase_rate=("ordered_next_30d", "mean"), median_orders=("orders_12m", "median"))
        .round(3)
    )

,n,purchase_rate,median_rev,median_orders,share
loyalty_tier,,,,,
Bronze,5400,0.204,1679.375,2.0,0.45
Silver,3600,0.387,7076.730,5.0,0.30
Gold,2160,0.579,17348.265,10.0,0.18
Platinum,840,0.775,46193.035,21.0,0.07


,n,purchase_rate,median_orders
industry_vertical,,,
Construction & Contracting,1636,0.257,2.0
Food & Beverage Processing,1684,0.526,8.0
General Manufacturing,1908,0.327,3.0
Machining & Auto Parts,2392,0.396,5.0
Metal Fabrication,2672,0.317,4.0
Plastics & Packaging,1708,0.395,5.0


,n,purchase_rate,median_orders
region,,,
Great Lakes,3541,0.378,4.0
Northeast,2251,0.326,3.0
South Central,1886,0.423,5.0
Southeast,2528,0.389,5.0
West,1794,0.304,3.0


,n,purchase_rate,median_orders
employee_band,,,
1-10,4144,0.246,2.0
11-50,4197,0.350,4.0
251+,968,0.632,11.0
51-250,2691,0.483,6.0


In [5]:
# 2d. Recency: does "days since last order" behave the way the old rule assumes?
bins = [-1, 30, 60, 90, 180, 365, 730]
display(
    df.groupby(pd.cut(df.days_since_last_order, bins), observed=True)
    .agg(n=("account_id", "size"), purchase_rate=("ordered_next_30d", "mean"))
    .round(3)
)

# 2e. Consumables vs. capital: Terrence says these behave differently
display(
    df.groupby(pd.qcut(df.consumables_revenue_share, 4), observed=True)
    .agg(median_orders=("orders_12m", "median"),
         median_aov=("avg_order_value", "median"),
         purchase_rate=("ordered_next_30d", "mean"))
    .round(3)
)

,n,purchase_rate
days_since_last_order,,
"(-1, 30]",4333,0.569
"(30, 60]",2021,0.401
"(60, 90]",1193,0.325
"(90, 180]",1922,0.234
"(180, 365]",1573,0.133
"(365, 730]",958,0.077


,median_orders,median_aov,purchase_rate
consumables_revenue_share,,,
"(-0.001, 0.344]",2.0,1872.570,0.209
"(0.344, 0.533]",3.0,1367.740,0.317
"(0.533, 0.718]",5.0,1042.100,0.399
"(0.718, 1.0]",8.0,737.725,0.542


In [6]:
# 2f. Engagement signals vs. outcome (quartiles, low to high)
for col in ["quote_requests_90d", "portal_logins_90d", "rep_contacts_90d", "email_open_rate_90d"]:
    q = pd.qcut(df[col].rank(method="first"), 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
    print(col, df.groupby(q, observed=True).ordered_next_30d.mean().round(3).to_dict())

quote_requests_90d {'Q1 low': 0.343, 'Q2': 0.344, 'Q3': 0.367, 'Q4 high': 0.413}
portal_logins_90d {'Q1 low': 0.212, 'Q2': 0.285, 'Q3': 0.381, 'Q4 high': 0.588}
rep_contacts_90d {'Q1 low': 0.24, 'Q2': 0.289, 'Q3': 0.366, 'Q4 high': 0.571}
email_open_rate_90d {'Q1 low': 0.338, 'Q2': 0.361, 'Q3': 0.378, 'Q4 high': 0.389}
